In [ ]:
# Importando as bibliotecas essenciais
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Carregando o dataset
# Certifique-se de que o arquivo nba_stats.csv está na mesma pasta que este notebook
df = pd.read_csv('nba_stats.csv')

# Visualizando as primeiras linhas para entender os dados
display(df.head())

### Definição do Problema e Justificativa

O objetivo deste projeto é prever a variável alvo **WinShares**, que representa uma estimativa estatística do número de vitórias com as quais um jogador contribuiu para sua equipe. 

Como a variável alvo é um valor numérico contínuo (e não uma categoria ou classe discreta), este é um problema de **Aprendizado Supervisionado de Regressão**.

In [ ]:
# 1. Separando a variável alvo (y) e as variáveis preditoras (X)
y = df['WinShares']
X = df.drop(columns=['WinShares'])

# 2. Filtrando para manter apenas as colunas numéricas em X (remove nomes e textos automaticamente)
X = X.select_dtypes(include=[np.number])

# 3. Tratamento de valores nulos (substituindo pela média da coluna para garantir estabilidade)
X = X.fillna(X.mean())
y = y.fillna(y.mean())

# 4. Aplicação da Técnica de Seleção de Atributos (SelectKBest)
# Vamos selecionar as 10 variáveis estatísticas que têm maior impacto no WinShares
seletor = SelectKBest(score_func=f_regression, k=10)
X_selecionado = seletor.fit_transform(X, y)

# 5. Identificando quais colunas foram escolhidas pelo algoritmo
atributos_escolhidos = X.columns[seletor.get_support()]
print("Os 10 atributos selecionados de forma automatizada foram:")
print(atributos_escolhidos)

Divisão dos Dados (Treino e Teste)
Conforme solicitado, vamos dividir nosso dataset. Separaremos 75% dos dados para o treinamento do algoritmo e os 25% restantes serão ocultados do modelo para servirem como nossa base de teste.

In [ ]:
# Dividindo a base selecionada: 75% para treino e 25% para teste
X_train, X_test, y_train, y_test = train_test_split(X_selecionado, y, test_size=0.25, random_state=42)

print(f"Quantidade de linhas para Treinamento (75%): {X_train.shape[0]}")
print(f"Quantidade de linhas para Teste (25%): {X_test.shape[0]}")

Treinamento do Modelo
Para este problema de regressão, escolhemos o algoritmo Random Forest Regressor da biblioteca scikit-learn. Ele será treinado exclusivamente com a base de treino (75%).

In [ ]:
# Instanciando o modelo Random Forest
modelo = RandomForestRegressor(n_estimators=100, random_state=42)

# Treinando o modelo APENAS com os dados de treino
modelo.fit(X_train, y_train)

print("O algoritmo foi treinado com sucesso!")

In [ ]:
# Realizando as predições para a base de teste (os 25% restantes)
predicoes = modelo.predict(X_test)

# Criando uma tabela lado a lado para comparar a Realidade vs a Previsão da Inteligência Artificial
resultados = pd.DataFrame({
    'Valor Real (WinShares)': y_test.values,
    'Previsão do Modelo': predicoes
})

print("Comparativo das 10 primeiras previsões na base de teste:")
display(resultados.head(10))

# Calculando as métricas de acerto
mse = mean_squared_error(y_test, predicoes)
r2 = r2_score(y_test, predicoes)

print(f"\nMétricas de Avaliação:")
print(f"Erro Quadrático Médio (MSE): {mse:.4f}")
print(f"R² Score: {r2:.4f} (Nota de 0 a 1 indicando a precisão do modelo)")

# Criando a sua própria IA: Parte II
Nesta segunda parte, iremos aprimorar o modelo desenvolvido anteriormente criando um **Pipeline**. A principal mudança é que a divisão da base de dados (Treino e Teste) ocorrerá **antes** da etapa de preparação dos dados, garantindo que não ocorra vazamento de dados (*data leakage*).

In [ ]:
# Importando as bibliotecas essenciais (incluindo o Pipeline e o SimpleImputer agora)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Carregar o dataset (o mesmo da parte I)
df = pd.read_csv('nba_stats.csv')

# Separando a variável alvo (y) e as variáveis preditoras (X)
y = df['WinShares']
X = df.drop(columns=['WinShares'])

# Mantendo apenas as colunas numéricas
X = X.select_dtypes(include=[np.number])

In [ ]:
# 2. Divisão do dataset entre treinamento (75%) e testes (25%) ANTES da preparação
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# --- CORREÇÃO AQUI ---
# Tratando os valores nulos da variável alvo 'y' manualmente, pois o Pipeline só trata o 'X'.
# Calculamos a média usando APENAS o y_train para evitar o Data Leakage
media_y_train = y_train.mean()

y_train = y_train.fillna(media_y_train)
y_test = y_test.fillna(media_y_train)
# ---------------------

print(f"Quantidade de linhas para Treinamento: {X_train.shape[0]}")
print(f"Quantidade de linhas para Teste: {X_test.shape[0]}")

In [ ]:
# 3. Criando o pipeline com a Preparação e o Treinamento
pipeline = Pipeline(steps=[
    # Preparação 1: Preencher valores nulos com a média (substitui o fillna manual)
    ('preenchimento_nulos', SimpleImputer(strategy='mean')),
    
    # Preparação 2: Seleção dos 10 melhores atributos (mesmo da Parte I)
    ('selecao_atributos', SelectKBest(score_func=f_regression, k=10)),
    
    # Treinamento: Algoritmo Random Forest (mesmo da Parte I)
    ('modelo', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Treinamento do pipeline (ele prepara os dados de treino e treina o algoritmo em sequência)
pipeline.fit(X_train, y_train)
print("Pipeline treinado com sucesso!")

In [ ]:
# 4. Mostrar as predições do pipeline para a base de testes
predicoes = pipeline.predict(X_test)

# 5. Calculando e mostrando as duas métricas escolhidas (MSE e R2)
mse = mean_squared_error(y_test, predicoes)
r2 = r2_score(y_test, predicoes)

print("--- Métricas de Avaliação na Base de Testes ---")
print(f"Erro Quadrático Médio (MSE): {mse:.4f}")
print(f"R² Score: {r2:.4f}")

### 📊 Conclusões da Equipe quanto às Métricas

Escolhemos duas métricas para avaliar nosso problema de regressão: o **Erro Quadrático Médio (MSE)** e o **R² Score**.

* **Os valores são bons ou ruins?** Os valores são matematicamente excelentes (quase perfeitos). O R² Score indica que o modelo explica quase 100% da variância dos dados, e o MSE (que mede o erro) está muito próximo de zero.

* **Por quê?** Ao analisarmos o contexto do basquete, a variável que queremos prever (`WinShares`) é composta pela soma direta de outras duas variáveis estatísticas presentes na base de dados (`OffensiveWinShares` e `DefensiveWinShares`). Como o nosso passo de preparação (`SelectKBest`) identificou essa forte correlação, ele as incluiu no modelo. Isso causou um cenário conhecido como "vazamento de dados" (*Data Leakage*), onde o modelo basicamente aprendeu a somar as duas colunas ao invés de prever um cenário incerto.

* **De que forma poderiam melhorar?** Para que a Inteligência Artificial tenha uma real utilidade preditiva para uma equipe técnica da NBA, a principal melhoria seria **remover as variáveis `OffensiveWinShares` e `DefensiveWinShares`** antes de passar os dados para o Pipeline. Assim, forçaríamos o algoritmo a tentar prever as vitórias baseando-se em métricas reais de jogo, como pontos, rebotes, tocos e assistências, o que tornaria o desafio estatístico muito mais robusto e realista.